# MDC Preprocessing vNext — research-grade pipeline

Combines every fix proven in prior runs (v0..v3 run 4) and adds defensive gates so a stale or mis-scaled npz can never silently corrupt a model run.

**Evidence-based decisions:**

| Decision | Source | Reason |
|---|---|---|
| `mean + max + std + flow_count` per bucket (~166 feat) | v2, Exp A (test AUC 0.72) | Burst & variability are critical |
| Random session split per container | Exp A | Removes temporal shift inversion |
| Benign-only fit (medians, variance, corr, clip, scalers) | Exp A | Avoids ~65% attack poisoning |
| `StandardScaler + clip ±10` flow + bucket | Exp A | RobustScaler exploded to 1e5 in v3 run 2 |
| `T=10`, `STRIDE=2`, `15s` buckets | v2, Exp A | Best separability so far |
| Short-gap fill ≤ 4 buckets with benign medians | Exp A | Long zero-pads hurt benign reconstruction |
| `ATTACK_FRAC_THRESHOLD=0.5` (majority labeling) | Exp A | Prevents diluted attack windows |
| **Drift baseline** (per-feature mean/std/quantiles of benign train) | New | Foundation for drift monitor |
| **Manifest gate** in npz (`scaler_type`, `bucket_agg`, scale stats) | New | Stops stale-npz misuse |

Outputs to `Module4_MDC/processed_vnext/`: `windows_vnext.npz`, `preproc_vnext.pkl`, `manifest_vnext.json`, `drift_baseline_vnext.npz`.

## 0. Setup & config

In [1]:
import sys, os, subprocess, gc, json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'kagglehub[pandas-datasets]', 'joblib'], check=False)
    OUTPUT_DIR = os.path.join(os.getcwd(), 'data', 'processed')
else:
    OUTPUT_DIR = os.path.normpath('../data/processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

VERSION       = 'vnext'
RANDOM_STATE  = 42
BENIGN_LABEL  = 0

TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15

MIN_FLOWS          = 10_000
GAP_THRESHOLD      = 60
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD     = 0.95

BUCKET_FREQ           = '15s'
BUCKET_AGG            = 'mean_max_std'   # locked: best evidence (Exp A 0.72 AUC)
WINDOW_SIZE           = 10               # 150s windows
STRIDE                = 2                # 30s hop
ATTACK_FRAC_THRESHOLD = 0.5              # majority-attack labeling
MAX_GAP_BUCKETS       = 4                # fill gaps <=4 buckets only
POST_SCALE_CLIP       = 10.0             # numerical safety

DROP_COLS = [
    'Flow ID', 'Dst IP', 'Timestamp',
    'Fwd URG Flags', 'Bwd URG Flags',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count',
]

print(f'preprocess_{VERSION} config')
print(f'  IN_COLAB         = {_IN_COLAB}')
print(f'  OUTPUT_DIR       = {OUTPUT_DIR}')
print(f'  bucket_agg       = {BUCKET_AGG}  (~3x stats x 55 = ~166 feat + flow_count)')
print(f'  window           = {WINDOW_SIZE} x {BUCKET_FREQ} = {WINDOW_SIZE*15}s')
print(f'  stride           = {STRIDE} x {BUCKET_FREQ} = {STRIDE*15}s')
print(f'  attack_frac_thr  = {ATTACK_FRAC_THRESHOLD}')
print(f'  max_gap_buckets  = {MAX_GAP_BUCKETS}')
print(f'  post_scale_clip  = +/-{POST_SCALE_CLIP}')

preprocess_vnext config
  IN_COLAB         = True
  OUTPUT_DIR       = /content/data/processed
  bucket_agg       = mean_max_std  (~3x stats x 55 = ~166 feat + flow_count)
  window           = 10 x 15s = 150s
  stride           = 2 x 15s = 30s
  attack_frac_thr  = 0.5
  max_gap_buckets  = 4
  post_scale_clip  = +/-10.0


## 1. Load raw data (Kaggle or local CSV)

In [2]:
from pathlib import Path

KAGGLE_DATASET = 'yigitsever/misuse-detection-in-containers-dataset'
KAGGLE_CSV     = 'MDC dataset.csv'

if _IN_COLAB:
    try:
        from google.colab import userdata
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
            v = userdata.get(k)
            if v: os.environ[k] = v
    except Exception:
        pass

import kagglehub
from kagglehub import KaggleDatasetAdapter
try:
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, KAGGLE_DATASET, KAGGLE_CSV)
except Exception:
    root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    candidates = sorted(root.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
    df = pd.read_csv(candidates[0])

df.columns = df.columns.str.strip()
print(f'Raw shape: {df.shape}   labels={sorted(df["Label"].unique())}')

100%|██████████| 552M/552M [00:05<00:00, 97.5MB/s]

Extracting files...


Raw shape: (3231475, 87)   labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


## 2. Container filter + timestamp + session assignment

In [3]:
counts   = df['Src IP'].value_counts()
keep_ips = counts[counts >= MIN_FLOWS].index.tolist()
df       = df[df['Src IP'].isin(keep_ips)].copy().reset_index(drop=True)
print(f'After container filter: {len(df):,} rows  /  {len(keep_ips)} containers')

df['ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['ts']).sort_values(['Src IP', 'ts']).reset_index(drop=True)

def assign_sessions(g, threshold=60):
    gap = g['ts'].diff().dt.total_seconds().fillna(0)
    sn  = (gap > threshold).cumsum()
    g['session_id'] = g['Src IP'].astype(str) + '_s' + sn.astype(str)
    return g

df = df.groupby('Src IP', group_keys=False).apply(assign_sessions, threshold=GAP_THRESHOLD)
df['time_gap_s'] = df.groupby('Src IP')['ts'].diff().dt.total_seconds()
print(f'Sessions: {df["session_id"].nunique():,}')
print(df.groupby('Src IP')['session_id'].nunique().to_string())

After container filter: 3,191,165 rows  /  6 containers
Sessions: 3,043
Src IP
10.16.0.4      577
10.16.0.5      584
10.16.0.6      503
10.16.0.61    1044
10.16.0.9       57
100.64.0.2     278


## 3. Session-level random split + benign-only fit setup

Random per-container split (TRAIN_RATIO=0.70) → train ∪ holdout. All subsequent statistics (median, variance, correlation, clip, scaler) are fit on **training benign flows only**.

In [4]:
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
gc.collect()

_ss = df[['Src IP', 'session_id', 'ts']].drop_duplicates('session_id').copy()
rng = np.random.default_rng(RANDOM_STATE)
train_sessions, holdout_sessions = set(), set()
for src_ip, grp in _ss.groupby('Src IP'):
    sess = grp['session_id'].tolist()
    n_train = max(1, int(len(sess) * TRAIN_RATIO))
    shuffled = rng.permutation(sess)
    train_sessions.update(shuffled[:n_train])
    holdout_sessions.update(shuffled[n_train:])
del _ss; gc.collect()

train_mask  = df['session_id'].isin(train_sessions)
meta_cols   = ['Src IP', 'session_id', 'Label', 'ts']
meta_train  = df.loc[train_mask, meta_cols].reset_index(drop=True)
meta_holdout= df.loc[~train_mask, meta_cols].reset_index(drop=True)

non_meta = [c for c in df.columns if c not in meta_cols + ['time_gap_s']]
df_train = df.loc[train_mask, non_meta].copy().reset_index(drop=True)
df_hold  = df.loc[~train_mask, non_meta].copy().reset_index(drop=True)
del df; gc.collect()

# Benign-only filter for transform fitting
benign_idx = meta_train['Label'].eq(BENIGN_LABEL).values
print(f'train flows: {len(df_train):,}   benign: {benign_idx.sum():,}  '
      f'({benign_idx.mean()*100:.1f}%)')
print(f'hold  flows: {len(df_hold):,}    holdout attack rate: {(meta_holdout["Label"] != 0).mean()*100:.1f}%')

train flows: 2,456,046   benign: 2,409,525  (98.1%)
hold  flows: 735,118    holdout attack rate: 28.8%


## 4. Inf/NaN/dtype hygiene

In [5]:
def to_numeric_safe(df_):
    for c in df_.columns:
        df_[c] = pd.to_numeric(df_[c], errors='coerce').astype(np.float32)
    return df_

df_train = to_numeric_safe(df_train)
df_hold  = to_numeric_safe(df_hold)

df_train = df_train.replace([np.inf, -np.inf], np.nan)
df_hold  = df_hold .replace([np.inf, -np.inf], np.nan)

# benign-train medians
train_medians = df_train[benign_idx].median(numeric_only=True).fillna(0.0).astype(np.float32)
df_train = df_train.fillna(train_medians)
df_hold  = df_hold .fillna(train_medians)
print(f'Filled NaNs with benign-train medians ({len(train_medians)} columns)')

Filled NaNs with benign-train medians (77 columns)


## 5. Variance threshold filter (fit on benign train)

In [6]:
feat_names_before_var = df_train.columns.tolist()
vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(df_train.loc[benign_idx].to_numpy(dtype=np.float32))
var_mask = vt.get_support()
keep_var = [c for c, ok in zip(feat_names_before_var, var_mask) if ok]
df_train = df_train[keep_var]; df_hold = df_hold[keep_var]
print(f'VarianceThreshold {VARIANCE_THRESHOLD} -> kept {df_train.shape[1]} / {len(feat_names_before_var)}')

VarianceThreshold 0.01 -> kept 76 / 77


## 6. Correlation filter (benign-train sample)

In [7]:
n_sample = min(100_000, int(benign_idx.sum()))
sample   = df_train.loc[benign_idx].sample(n=n_sample, random_state=RANDOM_STATE).astype(np.float32)
corr     = sample.corr(numeric_only=True).abs()
upper    = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr= [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
df_train = df_train.drop(columns=drop_corr, errors='ignore')
df_hold  = df_hold .drop(columns=drop_corr, errors='ignore')
del sample, corr, upper; gc.collect()
print(f'Correlation>{CORR_THRESHOLD}: dropped {len(drop_corr)}  ->  {df_train.shape[1]} cols')

Correlation>0.95: dropped 22  ->  54 cols


## 7. Clip bounds (IQR-3 / p99) — fit on benign train

In [8]:
df_tr_b = df_train.loc[benign_idx]
clip_bounds, n_iqr, n_p99 = {}, 0, 0
for col in df_train.columns:
    s = df_tr_b[col]
    if s.nunique(dropna=False) <= 1:
        clip_bounds[col] = (None, float(abs(s.iloc[0])) if len(s) else None, 'const_cap')
        n_p99 += 1; continue
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        clip_bounds[col] = (q1 - 3*iqr, q3 + 3*iqr, 'iqr'); n_iqr += 1
    else:
        p99 = float(s.quantile(0.99))
        clip_bounds[col] = (None, p99 if p99 > 0 else None, 'p99'); n_p99 += 1
for col, (lo, hi, st) in clip_bounds.items():
    if st == 'const_cap' and hi is not None:
        df_train[col] = df_train[col].clip(upper=hi); df_hold[col] = df_hold[col].clip(upper=hi)
    elif st in ('iqr', 'p99'):
        df_train[col] = df_train[col].clip(lower=lo, upper=hi)
        df_hold [col] = df_hold [col].clip(lower=lo, upper=hi)
print(f'IQR-clipped {n_iqr}, p99/const-clipped {n_p99}')

IQR-clipped 27, p99/const-clipped 27


## 8. Flow-level StandardScaler (benign train) + clip ±10
RobustScaler was discarded — on benign-only flows the IQR can be ≈ 0 → values ~1e5 (v3 run 2 evidence).

In [9]:
feat_cols_final = df_train.columns.tolist()
X_tr = df_train.to_numpy(dtype=np.float32, copy=True)
X_ho = df_hold .to_numpy(dtype=np.float32, copy=True)
del df_train, df_hold; gc.collect()

scaler_flow = StandardScaler().fit(X_tr[benign_idx])
X_tr = np.clip(scaler_flow.transform(X_tr), -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
X_ho = np.clip(scaler_flow.transform(X_ho), -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
print(f'flow scaler fit on {int(benign_idx.sum())} benign flows  -- '
      f'max|X|={max(np.abs(X_tr).max(), np.abs(X_ho).max()):.3f}')

df_tr_proc = pd.DataFrame(X_tr, columns=feat_cols_final, index=meta_train.index)
df_tr_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_train[['Src IP','session_id','Label','ts']].values
df_ho_proc = pd.DataFrame(X_ho, columns=feat_cols_final, index=meta_holdout.index)
df_ho_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_holdout[['Src IP','session_id','Label','ts']].values
del X_tr, X_ho; gc.collect()

flow scaler fit on 2409525 benign flows  -- max|X|=10.000


0

## 9. 15-second bucketing — mean + max + std + flow_count

In [10]:
def bucket_flows(df_proc, feature_cols, bucket_freq, benign_label, flow_count_max=None):
    df_proc = df_proc.copy()
    df_proc['ts']     = pd.to_datetime(df_proc['ts'])
    df_proc['bucket'] = df_proc['ts'].dt.floor(bucket_freq)
    g = ['Src IP', 'session_id', 'bucket']
    feat_agg = df_proc.groupby(g, observed=True)[feature_cols].agg(['mean','max','std'])
    feat_agg.columns = [f'{c}_{s}' for c, s in feat_agg.columns]
    feat_agg = feat_agg.reset_index()
    std_cols = [c for c in feat_agg.columns if c.endswith('_std')]
    feat_agg[std_cols] = feat_agg[std_cols].fillna(0.0)
    cnt = df_proc.groupby(g, observed=True).size().reset_index(name='_flow_count_raw')
    lbl = df_proc.groupby(g, observed=True)['Label'].apply(
        lambda x: 0 if (x == benign_label).all() else 1).reset_index(name='label')
    out = feat_agg.merge(cnt, on=g).merge(lbl, on=g)
    if flow_count_max is None:
        flow_count_max = float(out['_flow_count_raw'].quantile(0.99))
    out['flow_count'] = (out['_flow_count_raw'] / flow_count_max).clip(upper=1.0).astype(np.float32)
    out = out.drop(columns=['_flow_count_raw'])
    bucket_feature_cols = [c for c in out.columns if c not in g + ['label']]
    return out, flow_count_max, bucket_feature_cols

train_b, flow_count_max, bucket_feature_cols = bucket_flows(
    df_tr_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=None)
hold_b, _, _ = bucket_flows(
    df_ho_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=flow_count_max)
del df_tr_proc, df_ho_proc; gc.collect()

print(f'Train   buckets: {len(train_b):,}   attack: {train_b["label"].sum():,} '
      f'({train_b["label"].mean()*100:.1f}%)')
print(f'Holdout buckets: {len(hold_b):,}    attack: {hold_b["label"].sum():,} '
      f'({hold_b["label"].mean()*100:.1f}%)')
print(f'Features/bucket: {len(bucket_feature_cols)}  ({len(feat_cols_final)} flow cols x 3 + flow_count)')

Train   buckets: 55,836   attack: 17,701 (31.7%)
Holdout buckets: 22,281    attack: 7,768 (34.9%)
Features/bucket: 163  (54 flow cols x 3 + flow_count)


## 10. Short-gap fill + bucket-level p1/p99 clip + bucket scaler

In [11]:
bucket_fill = (train_b.loc[train_b['label']==0, bucket_feature_cols]
               .median().fillna(0.0).astype(np.float32))

def short_gap_fill(bdf, freq, feat_cols, fill_vals, max_gap=MAX_GAP_BUCKETS):
    f = pd.Timedelta(freq); span = max_gap * f
    def per_session(g):
        g = g.sort_values('bucket').drop_duplicates(subset=['bucket'])
        src, sid = g['Src IP'].iloc[0], g['session_id'].iloc[0]
        rows, prev = [], None
        for _, r in g.iterrows():
            if prev is not None:
                gap = r['bucket'] - prev
                if f < gap <= span + f:
                    for tb in pd.date_range(prev + f, r['bucket'] - f, freq=freq):
                        rec = {c: float(fill_vals[c]) for c in feat_cols}
                        rec.update({'Src IP': src, 'session_id': sid, 'bucket': tb, 'label': 0})
                        rows.append(rec)
            rows.append(r.to_dict()); prev = r['bucket']
        return pd.DataFrame(rows)
    return (bdf.groupby(['Src IP','session_id'], group_keys=False)
               .apply(per_session).reset_index(drop=True))

train_b = short_gap_fill(train_b, BUCKET_FREQ, bucket_feature_cols, bucket_fill)
hold_b  = short_gap_fill(hold_b , BUCKET_FREQ, bucket_feature_cols, bucket_fill)
print(f'After short-gap fill (<= {MAX_GAP_BUCKETS} buckets) -- '
      f'train: {len(train_b):,}  hold: {len(hold_b):,}')

# Bucket-level clip from benign train p1/p99
_b = train_b.loc[train_b['label']==0, bucket_feature_cols]
lo = _b.quantile(0.01).astype(np.float32); hi = _b.quantile(0.99).astype(np.float32)
for c in bucket_feature_cols:
    L, H = float(lo[c]), float(hi[c])
    if not (np.isfinite(L) and np.isfinite(H) and L < H): continue
    train_b[c] = train_b[c].clip(L, H); hold_b[c] = hold_b[c].clip(L, H)
del _b; gc.collect()

# Bucket-level StandardScaler on benign train buckets
scaler_bucket = StandardScaler().fit(
    train_b.loc[train_b['label']==0, bucket_feature_cols].to_numpy(dtype=np.float32))
for bdf in (train_b, hold_b):
    arr = scaler_bucket.transform(bdf[bucket_feature_cols].to_numpy(dtype=np.float32))
    bdf[bucket_feature_cols] = np.clip(arr, -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
print(f'Bucket scaler fit on benign train buckets  -- '
      f'max|X|={float(np.abs(train_b[bucket_feature_cols].values).max()):.3f}')

After short-gap fill (<= 4 buckets) -- train: 62,103  hold: 24,537
Bucket scaler fit on benign train buckets  -- max|X|=8.786


## 11. Sliding windows (T=10, stride=2) — benign-only train

In [12]:
def make_windows(bucket_df, W, S, feat_cols, thr=ATTACK_FRAC_THRESHOLD):
    Xs, Ys, Cs = [], [], []
    for (src, sid), grp in bucket_df.groupby(['Src IP','session_id'], observed=True):
        grp = grp.sort_values('bucket')
        arr = grp[feat_cols].to_numpy(dtype=np.float32)
        lbl = grp['label'].to_numpy()
        for k in range(0, len(arr) - W + 1, S):
            seg = lbl[k:k+W]
            Xs.append(arr[k:k+W])
            Ys.append(int((seg != 0).mean() >= thr))
            Cs.append(src)
    if not Xs:
        return (np.empty((0, W, len(feat_cols)), np.float32),
                np.empty(0, np.int8), np.empty(0, object))
    return np.asarray(Xs, np.float32), np.asarray(Ys, np.int8), np.asarray(Cs, object)

X_train_all, y_train_all, c_train_all = make_windows(train_b, WINDOW_SIZE, STRIDE, bucket_feature_cols)
X_hold,      y_hold,      c_hold      = make_windows(hold_b , WINDOW_SIZE, STRIDE, bucket_feature_cols)
del train_b, hold_b; gc.collect()

benign_w = y_train_all == BENIGN_LABEL
X_train  = X_train_all[benign_w]
y_train  = y_train_all[benign_w]
c_train  = c_train_all[benign_w]
del X_train_all, y_train_all, c_train_all; gc.collect()
print(f'Train benign windows : {X_train.shape}')
print(f'Holdout windows      : {X_hold.shape}  attack rate {y_hold.mean()*100:.1f}%')

Train benign windows : (16795, 10, 163)
Holdout windows      : (10306, 10, 163)  attack rate 38.2%


## 12. Stratified val / test split by (container × label)

Stratifying jointly by container and label keeps per-container coverage in **both** val and test instead of only stratifying by label (improvement vs v2).

In [13]:
from sklearn.model_selection import train_test_split
strat_key = pd.Series(c_hold.astype(str) + '|' + y_hold.astype(str))
vc = strat_key.value_counts()
_safe_key = strat_key.where(strat_key.map(vc).ge(2), other='_other')
X_val, X_test, y_val, y_test, c_val, c_test = train_test_split(
    X_hold, y_hold, c_hold,
    test_size=0.5, stratify=_safe_key, random_state=RANDOM_STATE)
del X_hold, y_hold, c_hold; gc.collect()
print(f'X_val  : {X_val.shape}   attack {y_val.mean()*100:.1f}%   containers={len(set(c_val.tolist()))}')
print(f'X_test : {X_test.shape}  attack {y_test.mean()*100:.1f}%   containers={len(set(c_test.tolist()))}')

X_val  : (5153, 10, 163)   attack 38.2%   containers=5
X_test : (5153, 10, 163)  attack 38.2%   containers=5


## 13. Pre-save validation gates

In [14]:
_xmax  = float(max(np.abs(X_train).max(), np.abs(X_val).max(), np.abs(X_test).max()))
_xmean = float(np.abs(X_train).mean())
_nan   = bool(np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any())

print('Window scale check')
print(f'  mean|X_train|={_xmean:.4f}  max|X|={_xmax:.4f}  nan={_nan}')
if _nan or _xmax > 50 or _xmean > 20:
    raise ValueError(f'BAD SCALE: re-run from \u00a78 StandardScaler + clip; max|X|={_xmax}')
if (y_train != 0).any():
    raise ValueError('Train windows must be benign-only')

overlap = train_sessions & holdout_sessions
assert len(overlap) == 0, 'Session leakage between train and holdout!'
print('Session leakage      : PASS')
print(f'Train benign-only    : PASS ({(y_train==0).all()})')
print(f'Holdout attack rate  : {y_val.mean()*100:.1f}% val  /  {y_test.mean()*100:.1f}% test')

Window scale check
  mean|X_train|=0.7010  max|X|=8.7861  nan=False
Session leakage      : PASS
Train benign-only    : PASS (True)
Holdout attack rate  : 38.2% val  /  38.2% test


## 14. Drift baseline (for downstream monitor)

Per-feature mean / std / quantiles of **benign train buckets** — used by `mdc_model_vNext` to compute PSI and KS distance on any new window stream (live, holdout, future capture).

In [15]:
F = X_train.shape[2]
flat_train = X_train.reshape(-1, F)
drift_quantiles = np.linspace(0.0, 1.0, 11)   # 0,0.1,..,1.0
drift_baseline = {
    'mean'      : flat_train.mean(axis=0).astype(np.float32),
    'std'       : flat_train.std(axis=0).astype(np.float32) + 1e-6,
    'q'         : np.quantile(flat_train, drift_quantiles, axis=0).astype(np.float32),
    'q_levels'  : drift_quantiles.astype(np.float32),
    'feat_names': bucket_feature_cols,
}
print(f'drift baseline: features={F}  quantile levels={len(drift_quantiles)}')
print(f'  mean range  [{drift_baseline["mean"].min():.3f}, {drift_baseline["mean"].max():.3f}]')
print(f'  std  range  [{drift_baseline["std"].min():.3f},  {drift_baseline["std"].max():.3f}]')

drift baseline: features=163  quantile levels=11
  mean range  [-0.069, 0.111]
  std  range  [0.781,  1.089]


## 15. Save artifacts (windows, preproc, drift baseline, manifest)

In [16]:
import joblib

npz_path = f'{OUTPUT_DIR}/windows_{VERSION}.npz'
np.savez_compressed(
    npz_path,
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_val=y_val, y_test=y_test,
    c_val=c_val.astype(str), c_test=c_test.astype(str),
)
print(f'Saved windows  -> {npz_path}  ({os.path.getsize(npz_path)/1024/1024:.1f} MB)')

drift_path = f'{OUTPUT_DIR}/drift_baseline_{VERSION}.npz'
np.savez_compressed(drift_path, **drift_baseline)
print(f'Saved drift    -> {drift_path}')

pkl_path = f'{OUTPUT_DIR}/preproc_{VERSION}.pkl'
joblib.dump({
    'version': VERSION, 'created_at': datetime.now(timezone.utc).isoformat(),
    'random_state': RANDOM_STATE,
    'window_size': WINDOW_SIZE, 'stride': STRIDE, 'bucket_freq': BUCKET_FREQ,
    'bucket_agg': BUCKET_AGG, 'attack_frac_threshold': ATTACK_FRAC_THRESHOLD,
    'max_gap_buckets': MAX_GAP_BUCKETS, 'post_scale_clip': POST_SCALE_CLIP,
    'scaler_flow': scaler_flow, 'scaler_bucket': scaler_bucket,
    'train_medians': train_medians,
    'feat_names_before_var': feat_names_before_var,
    'keep_var_mask': var_mask, 'drop_corr': drop_corr,
    'clip_bounds': clip_bounds, 'feat_cols_final': feat_cols_final,
    'bucket_feature_cols': bucket_feature_cols,
    'flow_count_max': flow_count_max, 'bucket_fill_values': bucket_fill,
}, pkl_path)
print(f'Saved preproc  -> {pkl_path}')

manifest = {
    'version'              : VERSION,
    'created_at'           : datetime.now(timezone.utc).isoformat(),
    'random_state'         : RANDOM_STATE,
    'scaler_type'          : 'StandardScaler+clip(flow & bucket)',
    'bucket_agg'           : BUCKET_AGG,
    'post_scale_clip'      : POST_SCALE_CLIP,
    'attack_frac_threshold': ATTACK_FRAC_THRESHOLD,
    'max_gap_buckets'      : MAX_GAP_BUCKETS,
    'shapes': {
        'X_train': list(X_train.shape),
        'X_val'  : list(X_val.shape),
        'X_test' : list(X_test.shape),
    },
    'attack_rates'         : {'val': float(y_val.mean()), 'test': float(y_test.mean())},
    'window_size'          : WINDOW_SIZE,
    'stride'               : STRIDE,
    'bucket_freq'          : BUCKET_FREQ,
    'features_per_bucket'  : len(bucket_feature_cols),
    'scale_stats'          : {'mean_abs': _xmean, 'max_abs': _xmax, 'nan': _nan},
    'leakage_free'         : True,
}
with open(f'{OUTPUT_DIR}/manifest_{VERSION}.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print(f'Saved manifest -> {OUTPUT_DIR}/manifest_{VERSION}.json')

Saved windows  -> /content/data/processed/windows_vnext.npz  (26.4 MB)
Saved drift    -> /content/data/processed/drift_baseline_vnext.npz
Saved preproc  -> /content/data/processed/preproc_vnext.pkl
Saved manifest -> /content/data/processed/manifest_vnext.json


## 16. Export to Drive `processed_vnext/`

In [17]:
import shutil
if _IN_COLAB:
    from google.colab import drive
    _dr = '/content/drive'
    if not os.path.isdir(os.path.join(_dr, 'MyDrive')):
        os.makedirs(_dr, exist_ok=True)
        drive.mount(_dr)
    dst = Path('/content/drive/MyDrive/Module4_MDC/processed_vnext')
    dst.mkdir(parents=True, exist_ok=True)
    for name in (f'windows_{VERSION}.npz', f'preproc_{VERSION}.pkl',
                 f'drift_baseline_{VERSION}.npz', f'manifest_{VERSION}.json'):
        src = Path(OUTPUT_DIR) / name
        if src.is_file():
            shutil.copy2(src, dst / name)
            print(f'Copied -> {dst / name}')
else:
    print(f'Local only: artifacts in {OUTPUT_DIR}')

Mounted at /content/drive
Copied -> /content/drive/MyDrive/Module4_MDC/processed_vnext/windows_vnext.npz
Copied -> /content/drive/MyDrive/Module4_MDC/processed_vnext/preproc_vnext.pkl
Copied -> /content/drive/MyDrive/Module4_MDC/processed_vnext/drift_baseline_vnext.npz
Copied -> /content/drive/MyDrive/Module4_MDC/processed_vnext/manifest_vnext.json
